# AI_EXTRACT — Web Server Log Analysis

`AI_EXTRACT` pulls structured fields from unstructured or semi-structured text.

| Item | Detail |
|---|---|
| **Function** | `SNOWFLAKE.CORTEX.AI_EXTRACT` |
| **Data** | 200 synthetic JSON web server logs (15 hand-crafted + 185 generated) |
| **Use-case** | Extract structured fields from messy log data at scale |


## Step 1 — Environment & Table Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

In [ ]:
CREATE OR REPLACE TABLE web_server_logs (
    log_id      INT AUTOINCREMENT,
    raw_log     VARIANT,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

## Step 2 — Insert Log Data

15 realistic hand-crafted logs plus 185 generated entries.

In [ ]:
INSERT INTO web_server_logs (raw_log) VALUES
(PARSE_JSON('{"ip":"192.168.1.42","method":"GET","path":"/api/v2/users","status":200,"bytes":1420,"user_agent":"Mozilla/5.0","response_time_ms":45}')),
(PARSE_JSON('{"ip":"10.0.3.15","method":"POST","path":"/api/v2/orders","status":201,"bytes":890,"user_agent":"Python-Requests/2.28","response_time_ms":120}')),
(PARSE_JSON('{"ip":"172.16.0.8","method":"GET","path":"/api/v2/products/search?q=laptop","status":200,"bytes":15600,"user_agent":"Chrome/119.0","response_time_ms":230}')),
(PARSE_JSON('{"ip":"192.168.1.100","method":"DELETE","path":"/api/v2/sessions/abc123","status":204,"bytes":0,"user_agent":"curl/8.1","response_time_ms":15}')),
(PARSE_JSON('{"ip":"10.0.1.55","method":"PUT","path":"/api/v2/users/profile","status":500,"bytes":340,"user_agent":"Mobile-App/3.2","response_time_ms":5200}')),
(PARSE_JSON('{"ip":"192.168.2.30","method":"GET","path":"/api/v2/inventory","status":200,"bytes":28900,"user_agent":"DataDog-Agent/7.40","response_time_ms":89}')),
(PARSE_JSON('{"ip":"10.0.0.1","method":"POST","path":"/api/v2/auth/login","status":401,"bytes":120,"user_agent":"Mozilla/5.0","response_time_ms":33}')),
(PARSE_JSON('{"ip":"172.16.5.20","method":"GET","path":"/api/v2/reports/daily","status":200,"bytes":45000,"user_agent":"Tableau/2023.3","response_time_ms":1500}')),
(PARSE_JSON('{"ip":"192.168.1.42","method":"PATCH","path":"/api/v2/orders/9876/status","status":200,"bytes":210,"user_agent":"Internal-Service/1.0","response_time_ms":67}')),
(PARSE_JSON('{"ip":"10.0.2.200","method":"GET","path":"/health","status":200,"bytes":15,"user_agent":"kube-probe/1.27","response_time_ms":2}')),
(PARSE_JSON('{"ip":"192.168.3.75","method":"POST","path":"/api/v2/uploads","status":413,"bytes":0,"user_agent":"Chrome/119.0","response_time_ms":50}')),
(PARSE_JSON('{"ip":"10.0.1.55","method":"GET","path":"/api/v2/users","status":429,"bytes":80,"user_agent":"Script/batch-job","response_time_ms":5}')),
(PARSE_JSON('{"ip":"172.16.0.8","method":"GET","path":"/api/v2/products/12345","status":304,"bytes":0,"user_agent":"Chrome/119.0","response_time_ms":12}')),
(PARSE_JSON('{"ip":"192.168.1.200","method":"POST","path":"/api/v2/webhooks/stripe","status":200,"bytes":450,"user_agent":"Stripe-Webhook/1.0","response_time_ms":180}')),
(PARSE_JSON('{"ip":"10.0.4.10","method":"GET","path":"/api/v2/analytics/funnel","status":200,"bytes":8900,"user_agent":"Grafana/10.0","response_time_ms":670}'));

In [ ]:
INSERT INTO web_server_logs (raw_log)
SELECT PARSE_JSON(
    '{"ip":"10.0.' || UNIFORM(0,5,RANDOM())::STRING || '.' || UNIFORM(1,254,RANDOM())::STRING
    || '","method":"' || CASE UNIFORM(1,5,RANDOM()) WHEN 1 THEN 'GET' WHEN 2 THEN 'POST' WHEN 3 THEN 'PUT' WHEN 4 THEN 'DELETE' ELSE 'PATCH' END
    || '","path":"/api/v2/' || CASE UNIFORM(1,6,RANDOM()) WHEN 1 THEN 'users' WHEN 2 THEN 'orders' WHEN 3 THEN 'products' WHEN 4 THEN 'inventory' WHEN 5 THEN 'reports' ELSE 'auth' END
    || '","status":' || CASE UNIFORM(1,10,RANDOM()) WHEN 1 THEN '500' WHEN 2 THEN '401' WHEN 3 THEN '404' WHEN 4 THEN '429' ELSE '200' END
    || ',"bytes":' || UNIFORM(50,50000,RANDOM())::STRING
    || ',"user_agent":"Agent-' || UNIFORM(1,20,RANDOM())::STRING || '","response_time_ms":' || UNIFORM(1,3000,RANDOM())::STRING || '}'
) FROM TABLE(GENERATOR(ROWCOUNT => 185));

In [ ]:
SELECT COUNT(*) AS total_logs FROM web_server_logs;

## Step 3 — Extract Fields from Log Messages

Use `AI_EXTRACT` to pull structured data from the raw JSON log text.

In [ ]:
SELECT
    log_id,
    raw_log:status::INT AS http_status,
    SNOWFLAKE.CORTEX.AI_EXTRACT(
        raw_log::STRING,
        ['endpoint_category', 'is_error', 'client_type']
    ) AS extracted
FROM web_server_logs
LIMIT 10;

## Step 4 — Error Pattern Analysis

In [ ]:
WITH extracted AS (
    SELECT
        log_id,
        raw_log:status::INT AS status_code,
        raw_log:path::STRING AS path,
        raw_log:response_time_ms::INT AS response_time_ms,
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            raw_log::STRING,
            ['error_severity', 'endpoint_category']
        ) AS ai_fields
    FROM web_server_logs
    WHERE raw_log:status::INT >= 400
)
SELECT
    ai_fields:endpoint_category::STRING AS endpoint,
    ai_fields:error_severity::STRING    AS severity,
    COUNT(*)                             AS error_count,
    AVG(response_time_ms)                AS avg_response_ms
FROM extracted
GROUP BY endpoint, severity
ORDER BY error_count DESC;

## Key Takeaways

- `AI_EXTRACT` can infer field meanings from context without rigid parsing rules.
- Combine AI extraction with traditional JSON path access for best results.
- Great for logs, tickets, emails — anywhere structure is implicit.
- The field names you provide guide extraction — use descriptive names.
